# Train the tread wear model (good / moderate / worn)

Trains the model that estimates tread wear from a photo of the tread.

Data (both from Kaggle, so only the Kaggle key is needed):
* *Tyre Condition Classification Dataset*: smartphone photos from Indian tyre workshops, with NEW → `good` (~168), SERVICEABLE → `moderate` (~306), UNUSABLE → `worn` (~73)
* *Tire Tread Photos*: GoodTireTread → `good` (~234), BadTireTread → `worn` (~135)

That's about 900 images in total, so it's a small dataset. Adding the mechanic's own photos (see the optional upload cell) is the best way to improve accuracy.

Runtime: about 15–25 min on a free T4.

## 1. Setup
In Colab, go to **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

`REPO_URL` already points at this project on GitHub. If you're working without GitHub, upload the
project folder as a zip and unzip it to `/content/Tyre-defect-detection` instead.

In [ ]:
REPO_URL = "https://github.com/abhinav29890/TyreHealth.git"  # change this if you fork the repo

import os, subprocess
if not os.path.exists("/content/Tyre-defect-detection"):
    subprocess.run(["git", "clone", REPO_URL, "/content/Tyre-defect-detection"], check=True)
%cd /content/Tyre-defect-detection/training
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

### API keys (free)
* **Kaggle**: kaggle.com → Settings → *Create New Token*. In Colab, add secrets (🔑 icon on the left)
  named `KAGGLE_USERNAME` and `KAGGLE_KEY`, and turn on *Notebook access*.
* **Roboflow** is *optional*: it's only needed if you add a Roboflow source to `sources.json`. Use the secret `ROBOFLOW_API_KEY`.

In [ ]:
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "ROBOFLOW_API_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            if k != "ROBOFLOW_API_KEY":
                print(f"secret {k} not set")
except ImportError:
    print("Not on Colab: set KAGGLE_USERNAME / KAGGLE_KEY / ROBOFLOW_API_KEY as environment variables.")

## 2. Get the data
Downloads the sources listed in `training/sources.json`, maps them to our classes, and removes duplicates.

### Add your own photos (recommended)
Zip your photos into folders named after the classes and upload the zip with the cell below:
```
tread_photos.zip
├── good/      (new or nearly new tread, deep grooves)
├── moderate/  (visibly worn but grooves still clear)
└── worn/      (shallow grooves, wear indicators flush, bald patches)
```
Use the same kind of framing the mechanic will use in the app: close up, tread filling the frame.
Set `UPLOAD_OWN_PHOTOS = True` below to get an upload button. With it set to `False`, the cell does nothing,
so *Run all* won't stop here.

In [ ]:
UPLOAD_OWN_PHOTOS = False  # set to True to upload tread_photos.zip (good/ moderate/ worn/ folders)

import subprocess
if not UPLOAD_OWN_PHOTOS:
    print("Skipping own-photo upload.")
else:
    from google.colab import files
    up = files.upload()
    os.makedirs("data/own/tread", exist_ok=True)
    for name in up:
        subprocess.run(["unzip", "-q", "-o", name, "-d", "data/own/tread"], check=True)
    for d in sorted(os.listdir("data/own/tread")):
        print(d, len(os.listdir(os.path.join("data/own/tread", d))), "images")

In [ ]:
!python -m src.prepare_data --task tread --clean

Check that the images look right for each class. **Garbage in, garbage out.** If a source's folders are mapped wrongly, fix `sources.json` and re-run.

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

root = Path("data/tread")
classes = [d.name for d in sorted(root.iterdir()) if d.is_dir() and any(d.iterdir())]
fig, axes = plt.subplots(len(classes), 6, figsize=(12, 2.2 * len(classes)), squeeze=False)
for r, c in enumerate(classes):
    files = random.Random(0).sample(sorted((root / c).glob("*.jpg")), k=min(6, len(list((root / c).glob("*.jpg")))))
    for j, ax in enumerate(axes[r]):
        ax.axis("off")
        if j < len(files):
            ax.imshow(Image.open(files[j]))
        if j == 0:
            ax.set_title(c, loc="left")
plt.tight_layout()

## 3. Train
EfficientNet-B0 pretrained on ImageNet: the head is trained first, then the whole network is fine-tuned. The best epoch by validation macro-F1 is kept.

In [ ]:
!python -m src.train --task tread --data data/tread --epochs 30

## 4. Evaluate on the held-out test split

In [ ]:
!python -m src.evaluate --ckpt checkpoints/tread/best.pt --data data/tread
from IPython.display import Image as Show
Show("reports/tread_confusion.png")

## 5. Export for the browser
Exports to ONNX, checks it against PyTorch, then quantises to INT8. INT8 is used only if it agrees with fp32 on ≥98% of test images. It writes `web/models/<task>.onnx` and `<task>.json`.

In [ ]:
!python -m src.export_onnx --ckpt checkpoints/tread/best.pt --data data/tread

## 6. Get the model into the app
**Option A, download:** run the cell below, then copy the files into your local `web/models/` folder
(replacing the placeholder) and copy `reports/tread_*` into `training/reports/`. Commit and push.

**Option B, push from Colab:** commit `web/models` and `training/reports` with a GitHub token.

In [ ]:
!cd .. && zip -q -r /content/tread_model.zip web/models/tread.onnx web/models/tread.json training/reports/tread_metrics.json training/reports/tread_confusion.png
try:
    from google.colab import files
    files.download("/content/tread_model.zip")
except ImportError:
    print("Zip written to /content/tread_model.zip")

Unzip it at the project root (it contains the `web/...` and `training/...` paths), then reload the app. The demo-mode banner disappears once both models are real.